## Instalando pacotes necessários

In [ ]:
#!uv pip install duckdb pandas
#!uv pip install python-dotenv
#!uv pip install boto3
#!uv pip install pyarrow
#!uv pip install plotly

Resolved 3 packages in 382ms                                         
⠙ Preparing packages... (0/2)                                                   
⠙ Preparing packages... (0/2)-------------------     0 B/9.44 MiB            
⠙ Preparing packages... (0/2)-------------------     0 B/9.44 MiB            
narwhals             ------------------------------     0 B/433.55 KiB
⠙ Preparing packages... (0/2)-------------------     0 B/9.44 MiB            
narwhals             ------------------------------ 16.00 KiB/433.55 KiB
⠙ Preparing packages... (0/2)-------------------     0 B/9.44 MiB            
narwhals             ------------------------------ 32.00 KiB/433.55 KiB
⠙ Preparing packages... (0/2)-------------------     0 B/9.44 MiB            
narwhals             ------------------------------ 48.00 KiB/433.55 KiB
⠙ Preparing packages... (0/2)-------------------     0 B/9.44 MiB            
narwhals             ------------------------------ 60.30 KiB/433.55 KiB
⠙ Preparing packag

## Importando as bibliotecas necessárias

In [ ]:
import duckdb
import pandas as pd
import os
import boto3
import re
import pyarrow
import plotly.express as px
import plotly.io as pio
from dotenv import load_dotenv
from datetime import datetime
from botocore.client import Config
from botocore.exceptions import NoCredentialsError, EndpointConnectionError, ClientError


load_dotenv()


#from dotenv import load_dotenv

True

## Busca informacoes no .env

In [2]:
ACCESS_KEY = os.getenv('MINIO_SENHA')
SECRET_KEY = os.getenv('MINIO_LOGIN')
MINIO_URL = os.getenv('MINIO_URL')

## Monta a conexao com o MinIO

In [3]:
def test_s3_connection():
    client = boto3.client('s3',
        endpoint_url=MINIO_URL,
        aws_access_key_id=ACCESS_KEY,
        aws_secret_access_key=SECRET_KEY,
        region_name='us-east-1'
    )
    
    try:
        # Ação que realmente testa a conexão
        response = client.list_buckets()
        response2 = client.list_objects_v2(Bucket='prata', Prefix='dados_relacionais/FAC2FTER')
        # 2. Acesse os buckets a partir da resposta (response)
        buckets = [b['Name'] for b in response['Buckets']]
        lista_objetos = [b['Key'] for b in response2['Contents']]

        padrao = r'^dados_relacionais/FAC2FTER/'
        lista_filtrada = [re.sub(padrao, '', arquivo) for arquivo in lista_objetos]

        info = f"Buckets existentes: {buckets}"
        return True, "Conexão bem-sucedida!", info,lista_filtrada
    except ClientError as e:
        return False, f"Erro de autenticação/permissão: {e}", "Não foram achados buckets","sem objetos"
    except Exception as e:
        return False, f"Erro de conexão: {e}", "Não foram achados buckets","sem objetos"

# Uso
success, message, bucket_info, objetos = test_s3_connection()
print(message)
if success:
    print(bucket_info)
    print("-----------------------------------------------------------")
    print('Dentro do endereco prata/dados_relacionais/FAC2FTER/ temos:')
    print("-----------------------------------------------------------")
    for tabela in objetos:
        print(tabela)

Conexão bem-sucedida!
Buckets existentes: ['bronze', 'ouro', 'prata']
-----------------------------------------------------------
Dentro do endereco prata/dados_relacionais/FAC2FTER/ temos:
-----------------------------------------------------------
areas_de_interesse_20260206_135811.parquet
atribuicoes_de_usuarios_20260210_131843.parquet
camadas_20260206_135811.parquet
chats_20260210_131843.parquet
comentarios_20260210_131843.parquet
matriz_sincronizacao_20260210_131843.parquet
messages_20260210_131843.parquet
operacoes_20260210_131843.parquet
pontos_de_interesse_20260210_131843.parquet
posicoes_de_forcas_amigas_20260210_131843.parquet


## Testa a conexao do DuckDB com o MinIO

In [10]:
PRATA_BUCKET = 'prata/dados_relacionais/FAC2FTER/'
DUCKDB_FILE = 'datalake_analytics.duckdb'

def init_duckdb():
    print("🦆 Conectando ao DuckDB com configuração avançada...")
    
    con = duckdb.connect(
        DUCKDB_FILE,
        read_only=False,
        config={
            "allow_unsigned_extensions": True
        }
    )
    
    # Instala e carrega o suporte a S3
    con.execute("INSTALL httpfs; LOAD httpfs;")
    
    # Configurações de acesso ao S3 (MinIO ou AWS)
    clean_endpoint = MINIO_URL.replace("http://", "").replace("https://", "")
    con.execute(f"SET s3_endpoint='{clean_endpoint}';")
    con.execute(f"SET s3_access_key_id='{ACCESS_KEY}';")
    con.execute(f"SET s3_secret_access_key='{SECRET_KEY}';")
    con.execute(f"SET s3_region='us-east-1';")
    
    use_ssl = "true" if MINIO_URL.startswith("https") else "false"
    con.execute(f"SET s3_use_ssl={use_ssl};")
    
    con.execute("SET s3_url_style='path';")
    
    print("✅ DuckDB configurado com acesso S3 (leitura e escrita).")
    return con

con=init_duckdb()

🦆 Conectando ao DuckDB com configuração avançada...
✅ DuckDB configurado com acesso S3 (leitura e escrita).


In [ ]:
# def test_duckdb_s3_connection(con):
#     #raw_path = f"s3://{PRATA_BUCKET}/dados_relacionais/FAC2FTER/"
#     try:
#         # 1. Garantir que o endpoint não tenha protocolos para o parâmetro s3_endpoint
#         clean_endpoint = MINIO_URL.replace("http://", "").replace("https://", "")
        
#         con.execute("INSTALL httpfs; LOAD httpfs;")
        
#         # Configurações essenciais para MinIO
#         con.execute(f"SET s3_endpoint='{clean_endpoint}';")
#         con.execute(f"SET s3_access_key_id='{ACCESS_KEY}';")
#         con.execute(f"SET s3_secret_access_key='{SECRET_KEY}';")
#         con.execute(f"SET s3_region='us-east-1';") # MinIO geralmente aceita qualquer uma, mas deve ser consistente
        
#         # IMPORTANTE: Se o seu MINIO_URL começa com http://, mantenha isso como false
#         use_ssl = "true" if MINIO_URL.startswith("https") else "false"
#         con.execute(f"SET s3_use_ssl={use_ssl};")
        
#         con.execute("SET s3_url_style='path';")

#         # 2. Limpeza do Path: Garantir que não existam barras duplas //
#         # Remove barras extras no final e garante uma estrutura limpa
#         path = f"s3://{PRATA_BUCKET}"
#         # Adiciona o wildcard para o glob
#         print(path)
#         #sql_path = path + "" + "*" if path.endswith("/") else path + "/" + "*"

#         sql_path = f"{path}*"

#         print(f"🔍 Testando acesso: {sql_path}")
        
#         # 3. Execução
#         result = con.execute(f"SELECT * FROM glob('{sql_path}')").fetchall()
        
#         print(f"✅ Conectado! Encontrados {len(result)} arquivos.")
        
#     except Exception as e:
#         print(f"❌ Erro detalhado: {type(e)._name_}")
#         print(f"📝 Mensagem: {e}")

# # Execução
# con = init_duckdb()
# test_duckdb_s3_connection(con)

🦆 Conectando ao DuckDB com configuração avançada...
s3://prata/dados_relacionais/FAC2FTER/
🔍 Testando acesso: s3://prata/dados_relacionais/FAC2FTER/*
✅ Conectado! Encontrados 10 arquivos.


## Extrai Valores e realiza Joins de teste

In [5]:
joins_to_do = [
    ("s3://prata/dados_relacionais/FAC2FTER/operacoes_20260210_131843.parquet", 
     "s3://prata/dados_relacionais/FAC2FTER/posicoes_de_forcas_amigas_20260210_131843.parquet", 
     "id", "operacaoID"),

    ("s3://prata/dados_relacionais/FAC2FTER/operacoes_20260210_131843.parquet", 
     "s3://prata/dados_relacionais/FAC2FTER/pontos_de_interesse_20260210_131843.parquet", 
     "id", "operacaoID"),

    ("s3://prata/dados_relacionais/FAC2FTER/operacoes_20260210_131843.parquet", 
     "s3://prata/dados_relacionais/FAC2FTER/atribuicoes_de_usuarios_20260210_131843.parquet", 
     "id", "operacaoID"),

    ("s3://prata/dados_relacionais/FAC2FTER/operacoes_20260210_131843.parquet", 
     "s3://prata/dados_relacionais/FAC2FTER/matriz_sincronizacao_20260210_131843.parquet", 
     "id", "operacaoID"),

    ("s3://prata/dados_relacionais/FAC2FTER/operacoes_20260210_131843.parquet", 
     "s3://prata/dados_relacionais/FAC2FTER/chats_20260210_131843.parquet", 
     "id", "operacaoID"),

    ("s3://prata/dados_relacionais/FAC2FTER/operacoes_20260210_131843.parquet", 
     "s3://prata/dados_relacionais/FAC2FTER/messages_20260210_131843.parquet", 
     "id", "operacaoID"),

    ("s3://prata/dados_relacionais/FAC2FTER/operacoes_20260210_131843.parquet", 
     "s3://prata/dados_relacionais/FAC2FTER/comentarios_20260210_131843.parquet", 
     "id", "operacaoID"),
]

In [ ]:
# def join_and_export(main, ref, key_main, key_ref,conexao):
#     # Nome das tabelas no DuckDB
#     table_main = "main_table"
#     table_ref = "ref_table"

#     # Carrega os arquivos Parquet no DuckDB
#     conexao.execute(f"CREATE OR REPLACE TABLE {table_main} AS SELECT * FROM read_parquet('{main}')")
#     conexao.execute(f"CREATE OR REPLACE TABLE {table_ref} AS SELECT * FROM read_parquet('{ref}')")

#     # Monta a query com os nomes corretos e chaves
#     qry = f"""
#         SELECT m.*, r.*
#         FROM {table_main} AS m
#         LEFT JOIN {table_ref} AS r
#           ON m.{key_main} = r.{key_ref}
#     """

#     try:
#         df = conexao.execute(qry).fetch_df()
        
#         # Gera o nome do arquivo de saída
#         main_table = main.split('/')[-1].split('_')[0]
#         ref_table = ref.split('/')[-1].split('_')[0]
#         out_name = f"s3://ouro/FAC2FTER/consolidado_{main_table}__{ref_table}.parquet"
        
#         # Salva o resultado
#         df.to_parquet(out_name, engine='pyarrow', compression=None, use_dictionary=False)
        
#         print(f"✅ Exportado join: {main_table} + {ref_table} → {out_name} (linhas: {len(df)})")
        
#     except Exception as e:
#         print(f"❌ Erro ao juntar {main} + {ref}: {e}")




In [ ]:
# def join_and_export(main, ref, key_main, key_ref, conexao):
#     # Nome das tabelas temporárias
#     table_main = "main_table"
#     table_ref = "ref_table"

#     # Carrega as views
#     conexao.execute(f"CREATE OR REPLACE VIEW {table_main} AS SELECT * FROM read_parquet('{main}')")
#     conexao.execute(f"CREATE OR REPLACE VIEW {table_ref} AS SELECT * FROM read_parquet('{ref}')")

#     # Define nomes de saída
#     main_name = main.split('/')[-1].split('_')[0]
#     ref_name = ref.split('/')[-1].split('_')[0]
#     out_path = f"s3://ouro/FAC2FTER/consolidado_{main_name}__{ref_name}.parquet"

#     # Query de processamento
#     qry = f"""
#         SELECT m., r.
#         FROM {table_main} AS m
#         LEFT JOIN {table_ref} AS r
#           ON m.{key_main} = r.{key_ref}
#     """

#     print(f"⏳ Processando {main_name} + {ref_name}...")
    
#     try:
#         # USA O PRÓPRIO DUCKDB PARA ESCREVER NO S3 (Usa as credenciais já configuradas)
#         conexao.execute(f"COPY ({qry}) TO '{out_path}' (FORMAT PARQUET)")
#         print(f"✅ Exportado com sucesso para: {out_path}")
        
#     except Exception as e:
        print(f"❌ Erro ao processar: {e}")

In [17]:
def join_and_export(main, ref, key_main, key_ref, conexao):
    # Nome das tabelas temporárias
    table_main_name = main
    table_ref_name = ref

    # LIMPEZA: Remove tabelas ou views antigas para evitar conflito de tipos
    conexao.execute(f"DROP TABLE IF EXISTS {table_main_name}")
    conexao.execute(f"DROP VIEW IF EXISTS {table_main_name}")
    
    conexao.execute(f"DROP TABLE IF EXISTS {table_ref_name}")
    conexao.execute(f"DROP VIEW IF EXISTS {table_ref_name}")

    # AGORA SIM: Cria as novas Views
    conexao.execute(f"CREATE OR REPLACE VIEW {table_main_name} AS SELECT * FROM read_parquet('{main}')")
    conexao.execute(f"CREATE OR REPLACE VIEW {table_ref_name} AS SELECT * FROM read_parquet('{ref}')")

    # Define nomes de saída baseados nos arquivos originais
    main_file_prefix = main.split('/')[-1].split('_')[0]
    ref_file_prefix = ref.split('/')[-1].split('_')[0]
    out_path = f"s3://ouro/FAC2FTER/consolidado_{main_file_prefix}__{ref_file_prefix}.parquet"

    # Query de processamento
    qry = f"""
        SELECT m.*, r.*
        FROM {table_main_name} AS m
        LEFT JOIN {table_ref_name} AS r
          ON m.{key_main} = r.{key_ref}
    """

    print(f"⏳ Processando {main_file_prefix} + {ref_file_prefix}...")
    
    try:
        # Exporta direto para o S3
        conexao.execute(f"COPY ({qry}) TO '{out_path}' (FORMAT PARQUET)")
        print(f"✅ Exportado com sucesso para: {out_path}")
        
    except Exception as e:
        print(f"❌ Erro ao processar: {e}")

In [24]:
def join_and_export(main_path, ref_path, key_main, key_ref, conexao):
    # Nomes temporários
    temp_table_main = "temp_view_main"
    temp_table_ref = "temp_view_ref"

    # --- BLOCO DE LIMPEZA BLINDADO ---
    # Tenta apagar como VIEW. Se falhar, ignora.
    try: conexao.execute(f"DROP VIEW IF EXISTS {temp_table_main}")
    except: pass
    
    # Tenta apagar como TABLE. Se falhar, ignora.
    try: conexao.execute(f"DROP TABLE IF EXISTS {temp_table_main}")
    except: pass

    # Repete para a tabela de referência
    try: conexao.execute(f"DROP VIEW IF EXISTS {temp_table_ref}")
    except: pass
    try: conexao.execute(f"DROP TABLE IF EXISTS {temp_table_ref}")
    except: pass
    # ---------------------------------

    # 2. Cria as Views apontando para os arquivos na Prata
    conexao.execute(f"CREATE VIEW {temp_table_main} AS SELECT * FROM read_parquet('{main_path}')")
    conexao.execute(f"CREATE VIEW {temp_table_ref} AS SELECT * FROM read_parquet('{ref_path}')")

    # 3. Define nome do arquivo de saída
    main_name = main_path.split('/')[-1].split('_')[0]
    ref_name = ref_path.split('/')[-1].split('_')[0]
    out_path = f"s3://ouro/FAC2FTER/consolidado_{main_name}__{ref_name}.parquet"

    # 4. Processamento
    qry = f"""
        SELECT m.*, r.*
        FROM {temp_table_main} AS m
        LEFT JOIN {temp_table_ref} AS r
          ON m.{key_main} = r.{key_ref}
    """

    print(f"⏳ Processando: {main_name} + {ref_name}...")
    
    try:
        conexao.execute(f"COPY ({qry}) TO '{out_path}' (FORMAT PARQUET)")
        print(f"✅ Salvo na Ouro: {out_path}")
    except Exception as e:
        print(f"❌ Erro ao processar: {e}")

In [25]:
# Iteramos apenas sobre os 4 itens que existem na lista 'joins_to_do'
for a, b, ka, kb in joins_to_do:
    try:
        # Passamos a conexão 'con' explicitamente aqui na chamada da função
        join_and_export(a, b, ka, kb, conexao=con)
    except Exception as e:
        print(f"Erro ao juntar {a} + {b}: {e}")

⏳ Processando: operacoes + posicoes...
✅ Salvo na Ouro: s3://ouro/FAC2FTER/consolidado_operacoes__posicoes.parquet
⏳ Processando: operacoes + pontos...
✅ Salvo na Ouro: s3://ouro/FAC2FTER/consolidado_operacoes__pontos.parquet
⏳ Processando: operacoes + atribuicoes...
✅ Salvo na Ouro: s3://ouro/FAC2FTER/consolidado_operacoes__atribuicoes.parquet
⏳ Processando: operacoes + matriz...
✅ Salvo na Ouro: s3://ouro/FAC2FTER/consolidado_operacoes__matriz.parquet
⏳ Processando: operacoes + chats...
✅ Salvo na Ouro: s3://ouro/FAC2FTER/consolidado_operacoes__chats.parquet
⏳ Processando: operacoes + messages...
✅ Salvo na Ouro: s3://ouro/FAC2FTER/consolidado_operacoes__messages.parquet
⏳ Processando: operacoes + comentarios...
✅ Salvo na Ouro: s3://ouro/FAC2FTER/consolidado_operacoes__comentarios.parquet


## Visualizando uma das tabelas

In [26]:
# Caminho do arquivo que você gerou na pasta Ouro
arquivo_gerado = "s3://ouro/FAC2FTER/consolidado_operacoes__posicoes.parquet"

# Usa o DuckDB para ler do S3 (aproveitando as credenciais) e converte para Pandas
df_example = con.execute(f"SELECT * FROM read_parquet('{arquivo_gerado}') LIMIT 5").df()

print(df_example)

                        _airbyte_raw_id            _airbyte_extracted_at  \
0  019c47b4-4ae0-78c6-8a05-8e5bfcdb2be3 2026-02-10 10:18:42.363000-03:00   
1  019c47b4-4ae0-78c6-8a05-8e5bfcdb2be3 2026-02-10 10:18:42.363000-03:00   
2  019c47b4-4ae0-78c6-8a05-8e5bfcdb2be3 2026-02-10 10:18:42.363000-03:00   
3  019c47b4-4ae0-78c6-8a05-8e5bfcdb2be3 2026-02-10 10:18:42.363000-03:00   
4  019c47b4-4ae0-78c6-8a05-8e5bfcdb2be3 2026-02-10 10:18:42.363000-03:00   

                    _airbyte_meta  _airbyte_generation_id  \
0  {'sync_id': 12, 'changes': []}                       1   
1  {'sync_id': 12, 'changes': []}                       1   
2  {'sync_id': 12, 'changes': []}                       1   
3  {'sync_id': 12, 'changes': []}                       1   
4  {'sync_id': 12, 'changes': []}                       1   

                                     id                      nome  \
0  1bd3cbb8-97c8-4e4d-be75-ae5d56d3d9b1  Exercício Operação ATLAS   
1  1bd3cbb8-97c8-4e4d-be75-ae5d56d3d9b

In [27]:
# Caminhos dos arquivos no S3
path_operacoes = "s3://prata/dados_relacionais/FAC2FTER/operacoes_20260210_131843.parquet"
path_posicoes  = "s3://prata/dados_relacionais/FAC2FTER/posicoes_de_forcas_amigas_20260210_131843.parquet"

# 1. CRIA A TABELA NA MEMÓRIA DO DUCKDB
# Substituímos o 'pg.public...' pelo 'read_parquet'
print("⏳ Criando tabela 'operacoes_com_posicoes'...")

con.execute(f"""
    CREATE OR REPLACE TABLE operacoes_com_posicoes AS 
    SELECT * FROM (SELECT * FROM read_parquet('{path_operacoes}')) o 
    LEFT JOIN read_parquet('{path_posicoes}') p 
    ON o.id = p.operacaoID
""")
print("✅ Tabela criada na memória!")


# 2. EXPORTA A TABELA CONSOLIDADA (COMO VOCÊ PEDIU)
# Agora podemos usar o nome da tabela simples que acabamos de criar
print("⏳ Exportando para CSV...")

# Se quiser salvar LOCALMENTE no Jupyter:
con.execute("COPY operacoes_com_posicoes TO 'operacoes_com_posicoes.csv' (HEADER, DELIMITER ',')")

# Se quiser salvar direto no S3 (Bucket Ouro), use esta linha no lugar da de cima:
# con.execute("COPY operacoes_com_posicoes TO 's3://ouro/operacoes_com_posicoes.csv' (HEADER, DELIMITER ',')")

print("✅ Tabela consolidada exportada: operacoes_com_posicoes.csv")

⏳ Criando tabela 'operacoes_com_posicoes'...
✅ Tabela criada na memória!
⏳ Exportando para CSV...
✅ Tabela consolidada exportada: operacoes_com_posicoes.csv


In [ ]:
# ==========================================================
# CONFIGURAÇÃO PARA FUNCIONAR EM QUALQUER AMBIENTE (WSL2, VSCode)
# ==========================================================

# Não usa MIME → Não usa nbformat → Não dá erro
pio.renderers.default = "browser"

# ==========================================================
# 1. CARREGAR O CSV
# ==========================================================

csv_file = "Tabelas soltas/export_pg_posicoes_de_forcas_amigas.csv"
csv_file = "Tabelas soltas/export_pg_posicoes_de_localizadores.csv"
df = pd.read_csv(csv_file)

print("Linhas carregadas:", len(df))

# ==========================================================
# 2. PARSER AUTOMÁTICO DO JSON (value)
# ==========================================================

def parse_value_auto(raw_json):
    try:
        obj = json.loads(raw_json)

        # ----------------------------------------
        # Tentativas automáticas de encontrar LAT/LON
        # ----------------------------------------
        lat = None
        lon = None

        # Caso 1: padrão FAC2FTer
        if isinstance(obj, dict):
            if "posicao" in obj:
                if "latLng" in obj["posicao"]:
                    lat = obj["posicao"]["latLng"].get("lat")
                    lon = obj["posicao"]["latLng"].get("lng")

        # Caso 2: lat/lon em nível superior
        if lat is None:
            lat = obj.get("lat") or obj.get("latitude")
        if lon is None:
            lon = obj.get("lon") or obj.get("lng") or obj.get("longitude")

        # ----------------------------------------
        # Detectar timestamp automaticamente
        # ----------------------------------------
        ts = None

        # CASO 1: timestamp em ms
        if "timestamp" in obj:
            try:
                ts = pd.to_datetime(obj["timestamp"], unit="ms")
            except:
                try:
                    ts = pd.to_datetime(obj["timestamp"])
                except:
                    pass

        # CASO 2: "ts"
        if ts is None and "ts" in obj:
            try:
                ts = pd.to_datetime(obj["ts"], unit="ms")
            except:
                try:
                    ts = pd.to_datetime(obj["ts"])
                except:
                    pass

        # CASO 3: datas em ISO
        for k, v in obj.items():
            if isinstance(v, str) and ("T" in v and ":" in v):
                try:
                    ts = pd.to_datetime(v)
                except:
                    pass

        return pd.Series({"lat": lat, "lon": lon, "timestamp": ts})

    except Exception as e:
        return pd.Series({"lat": None, "lon": None, "timestamp": None})


# ==========================================================
# 3. APLICAR O PARSER
# ==========================================================

df_parsed = df.join(df["value"].apply(parse_value_auto))
df_parsed = df_parsed.dropna(subset=["lat", "lon"])

print("Posições válidas encontradas:", len(df_parsed))

# ==========================================================
# 4. CRIAR TIMESTAMP ARTIFICIAL SE NÃO EXISTIR
# ==========================================================

if df_parsed["timestamp"].isna().all():
    print("⚠️ Nenhum timestamp encontrado → usando tempo artificial sequencial")
    df_parsed["timestamp"] = range(len(df_parsed))
else:
    print("⏳ Timestamp real encontrado")
    df_parsed["timestamp"] = df_parsed["timestamp"].fillna(method="ffill")
    df_parsed = df_parsed.sort_values("timestamp")

# ==========================================================
# 5. GERAR MAPA INTERATIVO COM TIMESLIDER
# ==========================================================

fig = px.scatter_map(
    df_parsed,
    lat="lat",
    lon="lon",
    hover_name="key",
    hover_data=["timestamp"],
    animation_frame="timestamp",
    zoom=5,
    height=750
)

# ==========================================================
# 6. ABRIR O MAPA NO NAVEGADOR DO WINDOWS
# ==========================================================

html_out = "mapa_interativo_localizadores.html"
pio.write_html(fig, html_out, auto_open=True)

print("\n🎉 Mapa criado com sucesso!")
print("Arquivo salvo em:", html_out)